# R-CNN — Rich feature hierarchies for accurate object detection (2013)

_Papers · Computer Vision_

**Propose a few regions, run a CNN on each, then classify and refine the box — detection by regions-with-CNN-features.**

---

This notebook is a practice scaffold for the **R-CNN — Rich feature hierarchies for accurate object detection (2013)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(0)

# ---- box helpers: corner form [x1,y1,x2,y2] and center form (cx,cy,w,h) ----
def area(b): return max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
def iou(a, b):
    ix = max(0.0, min(a[2],b[2]) - max(a[0],b[0]))
    iy = max(0.0, min(a[3],b[3]) - max(a[1],b[1]))
    inter = ix*iy
    union = area(a) + area(b) - inter
    return inter/union if union > 0 else 0.0
def to_center(b):  # corner -> (cx,cy,w,h)
    return ((b[0]+b[2])/2, (b[1]+b[3])/2, b[2]-b[0], b[3]-b[1])
def to_corner(c):  # (cx,cy,w,h) -> corner
    return [c[0]-c[2]/2, c[1]-c[3]/2, c[0]+c[2]/2, c[1]+c[3]/2]

# ---- 0. Sanity-check the lesson's worked example. ----
G = [1,1,6,6]; P = [3,3,8,7]
print("worked (a) IoU(P,G) =", round(iou(P, G), 4))          # 0.25
Pc, Gc = to_center(P), to_center(G)
tx = (Gc[0]-Pc[0]) / Pc[2]
ty = (Gc[1]-Pc[1]) / Pc[3]
tw = math.log(Gc[2] / Pc[2])
th = math.log(Gc[3] / Pc[3])
print("worked (b) targets t =", [round(tx,3), round(ty,3), round(tw,3), round(th,3)])  # [-0.4,-0.375,0.0,0.223]
# apply the transform (here d == t) and confirm we land on G
nx = Pc[0] + Pc[2]*tx; ny = Pc[1] + Pc[3]*ty
nw = Pc[2]*math.exp(tw); nh = Pc[3]*math.exp(th)
refined = to_corner((nx, ny, nw, nh))
print("refined box =", [round(v,2) for v in refined], " IoU after regression =", round(iou(refined, G), 4))  # 1.0


# ---- 1. A tiny synthetic image with one object (a bright square on the ground-truth box). ----
H = W = 32
img = torch.zeros(1, 1, H, W)
gt = [10, 8, 22, 24]                     # ground-truth object box (corner form)
img[0, 0, gt[1]:gt[3], gt[0]:gt[2]] = 1.0

# ---- 2. "Selective search": a few hand-placed candidate proposals. ----
proposals = [[10,8,22,24], [12,10,24,26], [2,2,12,12], [16,4,30,28]]

# ---- 3. A small CNN-style feature extractor (stands in for deep AlexNet R-CNN used). ----
feat = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(),
                     nn.AdaptiveAvgPool2d(1), nn.Flatten())   # -> 4-dim feature per region
clf  = nn.Linear(4, 1)                                        # linear "object vs background" scorer

# ---- 4. The R-CNN per-region loop: warp -> CNN -> classify -> IoU -> box-regress. ----
def warp(box, size=8):                                        # crop + resize to fixed input
    x1,y1,x2,y2 = [int(v) for v in box]
    crop = img[:, :, y1:y2, x1:x2]
    return nn.functional.interpolate(crop, size=(size, size), mode="bilinear", align_corners=False)

cnn_passes = 0
print("\nper-proposal pipeline:")
for P in proposals:
    f = feat(warp(P)); cnn_passes += 1                        # ONE CNN pass per proposal (the slow part)
    score = torch.sigmoid(clf(f)).item()
    j = iou(P, gt)
    label = "object" if j >= 0.5 else "background"            # R-CNN fine-tuning rule (IoU >= 0.5)
    # box-regression targets to pull this proposal onto the ground truth (Appendix C)
    Pc, Gc = to_center(P), to_center(gt)
    t = [(Gc[0]-Pc[0])/Pc[2], (Gc[1]-Pc[1])/Pc[3], math.log(Gc[2]/Pc[2]), math.log(Gc[3]/Pc[3])]
    refined = to_corner((Pc[0]+Pc[2]*t[0], Pc[1]+Pc[3]*t[1], Pc[2]*math.exp(t[2]), Pc[3]*math.exp(t[3])))
    print(f"  P={P}  score={score:.2f}  IoU={j:.2f} ({label})  "
          f"IoU after regression={iou(refined, gt):.2f}")

# ---- 5. The slowness that motivated Fast R-CNN: CNN passes scale with proposals * images. ----
print(f"\nCNN forward passes for THIS image: {cnn_passes} (one per proposal)")
proposals_per_image = 2000                                    # R-CNN's real count (paper, Sec 2.2)
for n_images in (1, 100, 5000):
    print(f"  {n_images} images x {proposals_per_image} proposals = "
          f"{n_images*proposals_per_image} CNN passes (R-CNN)  vs  {n_images} (Fast R-CNN, 1 per image)")
# R-CNN runs the CNN once per region with no sharing -> ~2000x the passes of Fast R-CNN.
# (Numbers here are our toy run, not the paper's reported timings.)

## Visualize the data & results

_How does R-CNN's CNN-forward-pass cost grow with dataset size, vs Fast R-CNN (one pass per image)?_

In [ ]:
# The cost model R-CNN's per-region loop implies (vs Fast R-CNN's shared pass).
proposals_per_image = 2000          # R-CNN, paper Sec 2.2 ("around 2000 region proposals")
for n_images in range(0, 101, 10):
    rcnn = n_images * proposals_per_image   # one CNN pass per proposal
    fast = n_images                         # Fast R-CNN: one CNN pass per image, then cheap crops
    print([n_images, rcnn, fast])
# [0, 0, 0]
# [10, 20000, 10]
# [20, 40000, 20]
# ...
# [100, 200000, 100]
# R-CNN's passes scale with proposals x images; Fast R-CNN's with images only (~2000x fewer).
# (Our toy cost model, not the paper's wall-clock timings.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The slowness ablation / Fast R-CNN motivation. R-CNN runs the CNN once per proposal. Fast
            R-CNN runs the CNN once per image and crops features per region. For an image with $N=2000$
            proposals, by what factor does Fast R-CNN cut the number of CNN forward passes, and why is the
            output quality not thrown away?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count R-CNN passes for one image: one warp-and-extract per proposal, so $N = 2000$ CNN forward passes. — _R-CNN shares no computation between overlapping proposals &mdash; each region is processed from scratch._
- Count Fast R-CNN passes: one pass over the whole image to make a feature map, then cheap per-region crops of that map (no extra CNN passes). — _Overlapping proposals reuse the same shared feature map, so the expensive convolutions run only once._
- Take the ratio: $2000 / 1 = 2000\times$ fewer CNN passes; accuracy holds because each region still gets its own feature (cropped from the shared map) and its own classifier + box regressor. — _The per-region features survive &mdash; only the redundant re-convolution is removed._

**Answer:** Fast R-CNN reduces ~2000 CNN forward passes per image to 1 (about a $2000\times$
                 cut), by running the convolutions once over the whole image and then cropping per-region
                 features from that shared feature map. Quality is preserved because every proposal still
                 receives its own feature vector and its own classifier + box-regression head &mdash; only the
                 wasteful re-computation on overlapping regions is removed. This redundancy is exactly the
                 slowness our notebook counts, and the reason Fast R-CNN exists.

</details>

**Problem 2.** A proposal $P=[2,2,6,5]$ (corner form) and ground truth $G=[1,1,5,5]$. Compute their IoU and decide
            whether $P$ is a fine-tuning positive for that object.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Areas: $P$ is $4\times3=12$; $G$ is $4\times4=16$. — _Width $=x_2-x_1$, height $=y_2-y_1$ in corner form._
- Intersection: $x$ from $\max(1,2)=2$ to $\min(5,6)=5$ &rarr; width $3$; $y$ from $\max(1,2)=2$ to $\min(5,5)=5$ &rarr; height $3$; area $=9$. — _Overlap is the box of the larger left/top edges and the smaller right/bottom edges._
- Union $=12+16-9=19$; IoU $=9/19\approx0.47$. Since $0.47 \lt 0.5$, $P$ is not a positive (it is background for fine-tuning). — _R-CNN's &sect;2.3 rule: IoU $\ge 0.5$ with a ground-truth box marks a proposal positive._

**Answer:** IoU $= 9/19 \approx 0.47$. Because that is just below the $0.5$ fine-tuning threshold, $P$
                 is labelled background, not a positive for the object &mdash; a near-miss the box
                 regressor would need to pull in before it counts as a detection.

</details>

**Problem 3.** Using the worked example's targets, apply the box regression. Proposal center/size is
            $P=(5.5,\,5.0,\,5,\,4)$ and the predicted transform equals the targets
            $(d_x,d_y,d_w,d_h)=(-0.4,\,-0.375,\,0,\,0.223)$. What box do you get, and what is its IoU with
            the ground truth $G=(3.5,3.5,5,5)$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Center: $x = P_x + P_w d_x = 5.5 + 5(-0.4) = 3.5$; $\;y = P_y + P_h d_y = 5.0 + 4(-0.375) = 3.5$. — _Apply (invert) the center targets: new center $= P_{x} + P_{w}\,d_x$._
- Size: $w = P_w e^{d_w} = 5 e^{0} = 5$; $\;h = P_h e^{d_h} = 4 e^{0.223} \approx 4 \times 1.25 = 5$. — _Apply the log-space size targets: new width $= P_w e^{d_w}$._
- Refined box has center $(3.5,3.5)$, size $(5,5)$ &mdash; identical to $G$, so IoU $=1.0$. — _When the predicted transform equals the targets, the proposal lands exactly on the ground truth._

**Answer:** The refined box is center $(3.5,3.5)$, size $(5,5)$ &mdash; exactly the ground truth
                 $G$, so IoU $=1.0$. The regression moved a poorly-fitting proposal onto the object, which is
                 why the paper reports box regression "boosting mAP by 3 to 4 points." Predicting the targets
                 $(t_x,t_y,t_w,t_h)$ and applying them via $P_x+P_w d_x$ and $P_w e^{d_w}$ recovers $G$.

</details>